# VWAP_V1.3 实盘/模拟账户接入（Binance Testnet 版）

你现在要做的是：把教学版原本的 `MockExchange` 替换为“真实交易所的适配器”，从而让 VWAP 执行器真的对 Binance **模拟账户（Testnet）**下单。

本仓库已经做了这一步：新增了 `BinanceSpotTestnetExchange` 适配器，并且 `run.py` 会根据配置里的 `exchange.adapter` 自动选择用哪个交易所。



## 1. 准备：设置 API Key（环境变量）

在终端（zsh）执行：

```bash
export EXCHANGE_API_KEY="你的binance_testnet_api_key"
export EXCHANGE_API_SECRET="你的binance_testnet_api_secret"
```

说明：
- 代码里不会写死 key
- 它会在 `vwap_executor/exchange/credentials.py` 里读取这两个环境变量
- `secrets/.env.example` 只是示例（程序不会自动加载它）


In [ ]:
# 检查环境变量是否存在
import os
print('EXCHANGE_API_KEY set:', bool(os.getenv('EXCHANGE_API_KEY')))
print('EXCHANGE_API_SECRET set:', bool(os.getenv('EXCHANGE_API_SECRET')))


## 2. 选择 Binance Testnet 适配器

你只需要改配置文件，而不需要动核心 VWAP 执行逻辑。

新增的示例配置在这里：
- `config/binance_spot_testnet_example_config.json`

它做了这些事：
- `instrument_type = spot`
- `exchange.adapter = binance_spot_testnet`
- `common` 里指定了 `symbol/side/notional/base_currency/quote_currency`



## 3. 运行命令（让执行立刻开始）

建议你先用短一些的 `total_duration_seconds` / `order_interval_seconds` 测试流程是否正常。

你可以直接运行示例：

```bash
python3 run.py --config config/binance_spot_testnet_example_config.json --start-offset-seconds 1
```

说明：
- `--start-offset-seconds 1` 会把配置里的 `common.start_time` 覆盖成 “当前时间 + 1 秒”，避免你等待很久。
- 程序会把订单明细写到 `execution_logs.jsonl`（由配置中的 `log_storage.output_jsonl_path` 控制）。


## 4. 适配器是怎么“用 API key 的”（接口层面）

`BinanceSpotTestnetExchange` 实现了 `vwap_executor/exchange/base.py` 的抽象接口：

1. `get_best_prices(symbol)`
   - 调用：`GET /api/v3/ticker/bookTicker`
   - 返回 `bid/ask`（用于计算限价 `price_offset`）

2. `get_available_base_qty(symbol)`
   - 调用（带签名）：`GET /api/v3/account`
   - 从 `balances` 里取你的 `base_currency` 的 `free`（用于现货 SELL 不超卖）

3. `place_limit_order(...)`
   - 调用（带签名）：`POST /api/v3/order`（LIMIT/GTC）
   - 然后轮询：`GET /api/v3/order` 获取本笔已成交部分
   - 若超时仍未完全成交，尝试撤单，返回已成交部分（这样执行器才能计算未成交比例并告警）

4. `place_market_order(...)`
   - 调用（带签名）：`POST /api/v3/order`（MARKET）
   - 返回本笔成交结果



## 5. 重要限制（你需要知道的点）

本教学版适配器把“Notional（计价金额）”换成 Binance 需要的 `quantity`，使用了：
- `quantity = notional / price`

因此会发生：
- 订单成交时的实际价格可能与下单估计价不同，所以 `notional` 可能不是完全等于输入 Notional
- 适配器会按交易所 `LOT_SIZE/stepSize` 向下取整数量，按 `PRICE_FILTER/tickSize` 向下取整价格，避免下单失败

此外：
- 当前实现的是 Spot Testnet（不是永续合约）
- 如果你要期货（perp），需要再实现对应的 `fapi` 接口适配器


## 6. 如果你想让 Notional 对齐得更精确

Binance 订单接口有些参数可以用 `quoteOrderQty` 直接按计价金额下单，但不同类型（LIMIT/MARKET、BUY/SELL）支持情况不同。

如果你希望我把适配器升级为：
- 用 `quoteOrderQty` 在支持的情况下直接按 Notional 下单
- 并提升未成交比例统计的准确性

你告诉我你使用的交易对（例如 BTCUSDT）和订单类型（LIMIT/MARKET）即可，我可以继续改。